# 데이터 결합 — merge · concat · join


**설명 전용 자료:** 개념·예제·주의점을 Markdown 셀로 제공합니다. 연습문제와 빈 풀이 셀은 같은 이름의 `_Practice.ipynb`에 있습니다.

**학습 목표:** 같은 구조의 데이터를 이어 붙이는 일과 공통 키로 정보를 붙이는 일을 구분하고, JOIN 방식에 따른 행 수·결측값 변화를 설명합니다.

## 처음 배우는 분을 위한 안내

키(key)는 두 표에서 같은 대상을 찾는 열입니다. customer_id는 고객 번호입니다. online·offline은 한 행이 주문 한 건, master·summary는 한 행이 고객 한 명입니다. concat은 표를 이어 붙이고 merge는 같은 키로 정보를 붙입니다. 인덱스는 행 이름표, NaN은 비어 있는 값입니다. many_to_one은 여러 주문에 고객 정보 한 행씩만 연결되는지 검사하는 규칙입니다.

**학습 순서:** 문제 1~6은 필수 기초, 7~12는 선택 도전입니다. 8~10·12번은 문제 2의 all_orders를, 10번은 9번의 master_dup을 씁니다. 앞 문제를 못 풀었다면 정답의 해당 풀이를 확인한 뒤 이어가세요.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

## 예제 데이터

아래 가상 데이터를 공통으로 사용합니다. 코드 블록은 읽기용이며 실행하려면 별도 코드 셀에 복사하세요. 예제는 순서대로 이어집니다. 독립 실행이 필요한 경우 이 준비 코드를 먼저 실행하세요.

```python
import pandas as pd

# 학습용 가상 데이터. 온라인·오프라인 각각 3행입니다.
online = pd.DataFrame({
    'order_id': [101, 102, 103], 'customer_id': [1, 2, 1],
    'revenue': [100, 200, 150], 'region': ['South', 'North', 'South']
})
offline = pd.DataFrame({
    'order_id': [104, 105, 106], 'customer_id': [3, 4, 2],
    'revenue': [300, 400, 250], 'region': ['East', 'West', 'North']
})
master = pd.DataFrame({
    'customer_id': [1, 2, 3, 5],
    'membership_tier': ['Gold', 'Silver', 'Bronze', 'Gold'],
    'region': ['South', 'North', 'East', 'Central']
})
summary = pd.DataFrame({
    'customer_id': [1, 2, 3, 4], 'total_orders': [2, 2, 1, 1],
    'total_spent': [250, 450, 300, 400]
})
store_a = pd.DataFrame({'product': ['A', 'B'], 'price': [100, 200]})
store_b = pd.DataFrame({'product': ['C', 'D'], 'quantity': [2, 3]})
product_info = pd.DataFrame({'product': ['A', 'B', 'C']}, index=[10, 20, 30])
sales_info = pd.DataFrame({'quantity': [3, 1, 2]}, index=[30, 10, 20])
```

## 결합 전 확인할 것

한 행이 주문인지 고객인지 먼저 확인합니다. 주문표의 customer_id는 반복될 수 있지만 고객 마스터는 고객마다 한 행이어야 합니다. 키의 자료형·결측값·중복을 확인한 뒤 결합합니다. 숫자 1과 문자열 "1"을 같은 키라고 가정하지 마세요.

| 방식 | 연결 기준 | 대표 상황 |
| --- | --- | --- |
| concat(axis=0) | 열 이름을 맞추며 행 추가 | 온라인·오프라인 주문 통합 |
| concat(axis=1) | 행 인덱스를 맞추며 열 추가 | 같은 인덱스의 속성표 연결 |
| merge | 지정한 키 값 | 주문에 고객 등급 추가 |
| join | 기본적으로 인덱스 | 고객 인덱스의 두 요약표 결합 |

concat은 중복 행을 자동 제거하지 않습니다. SQL UNION보다 UNION ALL에 가까운 연결이며, axis=1도 단순한 '현재 줄 순서'가 아니라 인덱스 정렬이 적용됩니다.

## concat: 행 방향 연결

axis=0은 아래에 행을 붙입니다. 기본 join='outer'이므로 열의 합집합이 남고 한쪽에 없는 값은 결측으로 표시됩니다. join='inner'는 공통 열만 남깁니다. ignore_index=True는 결과 축의 레이블을 재생성할 뿐 데이터 중복을 제거하지 않습니다. 기존 인덱스가 식별자로 필요하면 보존하는 편이 맞습니다.

```python
all_orders = pd.concat([online, offline], axis=0, ignore_index=True)
print(all_orders.shape)  # (6, 4)
different_columns = pd.concat([store_a, store_b], ignore_index=True)
print(different_columns.isna().sum())
```

## concat: 열 방향 연결

axis=1은 같은 인덱스 레이블끼리 맞춥니다. 아래 예제에서 A의 인덱스는 10이므로 quantity=1이 연결됩니다. 두 표가 동일 순서라는 보장이 있을 때만 양쪽 reset_index(drop=True) 후 위치에 맞춰 붙이세요.

```python
aligned = pd.concat([product_info, sales_info], axis=1)
print(aligned.loc[10])  # A, quantity 1
```

## merge와 JOIN 방식

키를 on으로 명시합니다. how의 기본값은 inner입니다.

| how | 유지하는 키 | 사용 목적 |
| --- | --- | --- |
| inner | 양쪽 공통 | 매칭되는 대상만 분석 |
| left | 왼쪽 모두 | 주문을 유지하며 고객 정보 추가 |
| right | 오른쪽 모두 | 오른쪽 기준 모집단 유지 |
| outer | 양쪽 모두 | 양쪽의 미매칭 대상까지 점검 |

왼쪽 행이 모두 유지된다는 말은 행 수가 반드시 같다는 뜻이 아닙니다. 오른쪽 키가 중복되면 왼쪽 한 행이 여러 행으로 늘 수 있습니다. pandas는 양쪽의 결측 키도 서로 매칭할 수 있어 일반적인 SQL의 NULL 처리와 차이가 있습니다.

```python
joined = all_orders.merge(master, on='customer_id', how='left',
                          suffixes=('_order', '_customer'),
                          validate='many_to_one', indicator=True)
print(joined['_merge'].value_counts())  # both 5, left_only 1
print(joined.shape[0])  # 6
```

## 겹치는 열과 결합 검증

양쪽에 region이 있지만 같은 뜻이라고 단정할 수 없습니다. 주문 지역과 고객 주요 지역은 다를 수 있으므로 suffixes로 구분합니다. on을 생략하면 같은 이름의 열들이 의도치 않게 함께 키가 될 수 있습니다. 이름이 다른 키는 left_on, right_on을 지정합니다.

validate='one_to_one'은 양쪽 키 유일성을, 'many_to_one'은 오른쪽 키 유일성을 확인합니다. indicator=True의 _merge는 both·left_only·right_only를 보여줍니다. 행 수뿐 아니라 고유 주문 수와 금액 합계도 비교해야 중복 집계를 발견하기 쉽습니다.

```python
assert len(joined) == len(all_orders)
assert joined['order_id'].nunique() == all_orders['order_id'].nunique()
assert joined['revenue'].sum() == all_orders['revenue'].sum()
```

## join: 인덱스 연결

join의 기본 how는 left로 merge의 기본값 inner와 다릅니다. 키를 인덱스로 맞추면 표현이 간단해집니다. join(on=...)을 사용하면 왼쪽 키 열과 오른쪽 인덱스를 연결할 수도 있으므로 '양쪽 키가 반드시 인덱스여야만 한다'는 제약은 아닙니다.

```python
customer_view = master.set_index('customer_id').join(
    summary.set_index('customer_id'), how='left', validate='one_to_one')
print(customer_view)  # 고객 5의 주문 요약은 결측
print(customer_view.index.is_unique)
```

## 선택 도전: 주문 한 건을 추가해 결합 점검하기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_order = pd.DataFrame({'order_id':[107], 'customer_id':[5], 'revenue':[120], 'region':['Central']})
```

### 진행 순서

준비 셀의 online·offline·master를 그대로 둡니다.
1. online, offline, extra_order를 행 방향으로 붙여 orders_extra에 저장합니다.
2. customer_id를 기준으로 master의 등급을 왼쪽 결합합니다. 주문 지역과 고객 지역은 suffixes로 구분합니다.
3. 행 수, 매출 합계, 등급이 비어 있는 주문 수를 출력합니다.

**완료 기준:** 7행, 매출 합계 1520, 등급 미매칭 주문 1개. 기존 1~12번의 6행·1400 기준은 바꾸지 않습니다.

### 선택 도전 풀이

```python
orders_extra = pd.concat([online, offline, extra_order], ignore_index=True)
checked_extra = orders_extra.merge(master, on='customer_id', how='left', suffixes=('_order','_customer'), validate='many_to_one')
print(len(checked_extra), checked_extra['revenue'].sum(), checked_extra['membership_tier'].isna().sum())
assert len(checked_extra)==7 and checked_extra['revenue'].sum()==1520
assert checked_extra['membership_tier'].isna().sum()==1
```

## 복습 순서

1. 개념과 입력·출력의 분석 단위를 확인합니다.
2. 작은 예제에서 결과를 손으로 예상합니다.
3. 연습용 노트북의 준비 셀을 실행하고 문제를 풉니다.
4. 결과 수치와 점검 기준을 비교하고 이유를 기록합니다.
5. 기본 문제를 마친 뒤 선택 도전을 진행합니다. 입력을 바꾸면 결과도 달라지는 이유를 설명합니다.